In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import *

# Parameters
hbar = 1
omega0 = 2*np.pi*200e3      # qubit frequency
omegac = omega0                # cavity frequency
Omega = 2*np.pi*10e3       # coupling strength
N = 100                         # number of cavity Fock states

n_fock = 5                     # Fock state

# ---------------------------------------------------------
# NEW: In the BS model, the frequency is strictly linear!
# Omega_n = Omega * n (No square root!)
# ---------------------------------------------------------
Omega_n_bs = Omega    # 2π * 10 kHz * 4 = 2π * 40 kHz
t_max = np.pi / (2*Omega_n_bs) # Time for exact peak in BS model

t_step = t_max / 10        
tlist = np.linspace(0, 2*t_max, 201)  

# Initial state: qubit in ground |0>
#psi0 = tensor(basis(2,1),(coherent(N,3)-coherent(N,-3)).unit())

#psi1 = tensor(basis(2,1),basis(N,1))
#psi2 = tensor(basis(2,1),basis(N,2))
#psi3 = tensor(basis(2,1),basis(N,3))
#psi4 = tensor(basis(2,1),basis(N,4))
psi1 = tensor(basis(2,1)*basis(2,1).dag(),(basis(N,1)*basis(N,1).dag()+basis(N,3)*basis(N,3).dag()).unit())
psi2 = tensor(basis(2,1)*basis(2,1).dag(),(basis(N,1)*basis(N,1).dag()+basis(N,2)*basis(N,2).dag()).unit())

# Operators
sm = 1/2*(sigmax()-1j*sigmay())
sm1 = tensor(sm, qeye(N))
sz1 = tensor(sigmaz(), qeye(N))

# ---------------------------------------------------------
# NEW: Constructing the Buck-Sukumar intensity-dependent operator
# ---------------------------------------------------------
# 1. Standard cavity annihilation operator
a_standard = destroy(N)

# 2. Create the sqrt(a_dag * a) matrix manually. 
# np.arange(N) creates an array [0, 1, 2, ..., N-1].
# np.diag puts them on the diagonal of a matrix.
# 1/ will give you a uniform coupling
sqrt_N = Qobj(np.diag(np.sqrt(np.arange(N))))

# 3. Multiply them to create the BS annihilation operator A = a * sqrt(N)
A_bs = a_standard * sqrt_N

# 4. Tensor it with the qubit identity operator for the joint space
a = tensor(qeye(2), A_bs)
# ---------------------------------------------------------

# Hamiltonian 
# (We use the newly defined 'a' in the interaction term exactly like before)
H_af = hbar * Omega * (a * sm1.dag() + a.dag() * sm1)
H_a = hbar * omega0 * sz1/2

# Note: The cavity self-energy remains based on standard photon number
H_f = hbar * omegac * tensor(qeye(2), num(N)) 
H = H_a + H_f + H_af

# Solve dynamics
output = mesolve(H, psi1, tlist, [], [])
output2 = mesolve(H, psi2, tlist, [], [])
#output3 = mesolve(H, psi3, tlist, [], [])
#output4 = mesolve(H, psi4, tlist, [], [])

# Qubit excited state probability P(|1>)
P_excited_1 = [expect(tensor(basis(2,0)*basis(2,0).dag(), qeye(N)), psi) for psi in output.states]
P_excited_2 = [expect(tensor(basis(2,0)*basis(2,0).dag(), qeye(N)), psi) for psi in output2.states]
#P_excited_3 = [expect(tensor(basis(2,0)*basis(2,0).dag(), qeye(N)), psi) for psi in output3.states]
#P_excited_4 = [expect(tensor(basis(2,0)*basis(2,0).dag(), qeye(N)), psi) for psi in output4.states]


gt_list = Omega * tlist

# Plot
fig = plt.figure(figsize=(10,6))
plt.plot(gt_list, P_excited_1, linewidth=2)
plt.plot(gt_list, P_excited_2, linewidth=2)
#plt.plot(gt_list, P_excited_3, linewidth=2)
#plt.plot(gt_list, P_excited_4, linewidth=2)
plt.xlabel("gt", fontsize=14)
plt.ylabel("Qubit |1> population", fontsize=14)
#plt.title("Exact Qubit Inversion in Buck-Sukumar Model with Odd Schrodinger Cat State |3>", fontsize=16)
plt.grid(False)
plt.show()


#fig.savefig("BS_Model 2.pdf", bbox_inches='tight')

In [ ]:
print(max(P_excited))

In [ ]:
# 1. Calculate gt (x-axis)
gt_list = Omega * tlist

# 2. Extract values as floats for all four Fock states
f_n1 = [float(val) for val in P_excited_1]
f_n2 = [float(val) for val in P_excited_2]
f_n3 = [float(val) for val in P_excited_3]
f_n4 = [float(val) for val in P_excited_4]

# 3. Stack into a table: [gt, n1, n2, n3, n4]
data = np.column_stack((gt_list, f_n1, f_n2, f_n3, f_n4))

# 4. Save to .dat file
np.savetxt("bs_model_fock_state_comparison.dat", data, 
           header="gt pop_n1 pop_n2 pop_n3 pop_n4", 
           comments='')

print("Data saved to bs_model_comparison.dat")

In [ ]:
# 1. Calculate gt (x-axis)
gt_list = Omega * tlist

# 2. Extract values as floats for the two mixtures
# P_excited_1 corresponds to (|1><1| + |3><3|) / 2 (Purely Odd)
# P_excited_2 corresponds to (|1><1| + |2><2|) / 2 (Mixed Parity)
f_mix_odd = [float(val) for val in P_excited_1]
f_mix_parity = [float(val) for val in P_excited_2]

# 3. Stack into a table: [gt, mixture_odd, mixture_mixed_parity]
data = np.column_stack((gt_list, f_mix_odd, f_mix_parity))

# 4. Save to .dat file
np.savetxt("bs_mixtures.dat", data, 
           header="gt pop_odd_mix pop_parity_mix", 
           comments='')

print("Data saved to bs_mixtures.dat")